# 03 · Smart Feedback: Adaptive Few-Shot Selection

The `"smart"` feedback strategy improves classification accuracy by selecting the **most relevant human corrections for each batch** of transcripts, rather than injecting the same static examples into every prompt.

**Why this matters:**
- A correction about a password reset issue is not helpful when classifying a billing dispute
- 10 corrections all pointing to the same taxonomy code waste prompt space — only the best representative is needed
- Boilerplate-heavy transcripts (templated agent responses, email chains) carry less classification signal than concise, varied ones

**Two-stage design:**

| Stage | When | What it does |
|---|---|---|
| **Stage 1 — Pool building** | Once at pipeline startup | Scores all corrections and keeps the best representative per corrected code |
| **Stage 2 — Per-batch reranking** | Before each batch's LLM call | From the pool, picks the examples most similar to the current batch's transcripts |

**No embeddings. No extra API calls.** Uses lightweight token-overlap similarity already present in the codebase.

---

**Notebook sections:**
1. Problem with static strategies
2. Stage 1 — content-aware pool building (scoring formula)
3. Stage 2 — per-batch Jaccard reranking
4. Strategy comparison (recent / diverse / frequent / smart)
5. Wiring into the pipeline (`RunConfig` + profile)
6. Bulk import from a reviewed spreadsheet
7. Diagnostic: inspecting what was selected

## 0 · Setup

In [ ]:
WORKSPACE = "_template"  # folder under use_cases/ to work in — change to run on another use case
import sys, os
from pathlib import Path

# Ensure project root is on sys.path and is the working directory
_root = Path().absolute()
# Navigate up until we find the rca_pipeline package
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Working directory: {os.getcwd()}")


In [ ]:
import json, tempfile, re, math
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pandas as pd

from rca_pipeline.enhancements.feedback import FeedbackStore, _tokenise

print("FeedbackStore imported ✓")

## 1 · The problem with static feedback strategies

With `strategy="recent"` or `strategy="diverse"`, the same few-shot block is sent with **every** batch in the run.

Consider this scenario: 10 human corrections, all correcting misclassified transcripts to the same code `CR3.1` (Account Recovery). With different strategies:

| Strategy | What goes into the prompt |
|---|---|
| `recent` | 5 of the 10 corrections (most recent) — all about account recovery |
| `diverse` | 1 correction (the most recent) — only one example despite 10 available |
| `frequent` | 1 correction (the most common pair) — same problem |
| `smart` | 1 correction (the best representative of CR3.1) **chosen based on content quality**, then surfaced only for batches that contain transcripts resembling account recovery issues |

The bigger problem: a batch of `CR1.2` (Login Blocked) transcripts still gets the CR3.1 few-shot examples injected, which can confuse the LLM.

In [ ]:
# Simulate the scenario: 10 corrections all pointing to CR3.1
tmp = tempfile.mkdtemp()
store = FeedbackStore(tmp)

base_ts = datetime(2026, 1, 1, tzinfo=timezone.utc)

# 10 CR3.1 corrections — varied quality
cr31_transcripts = [
    ("T001", "Customer called. Issue resolved. Thank you for contacting us.",          False, 0),
    ("T002", "Customer called. Issue resolved. Thank you for contacting us.",          False, 1),
    ("T003", "Customer called. Issue resolved. Thank you for contacting us.",          False, 2),
    ("T004", "Hi I forgot my username and cannot reset it via email anymore",           False, 3),
    ("T005", "Account locked after 3 failed attempts, recovery email not working",      True,  4),
    ("T006", "User cannot access account. Password reset link expired after 24h. "
             "Secondary email also unavailable. Escalated to Tier 2 for manual reset.", True,  5),
    ("T007", "Thank you for contacting support. I understand you are experiencing "
             "difficulties. Please follow these steps: 1. Go to login page "
             "2. Click forgot password 3. Check email 4. Click reset link",             False, 6),
    ("T008", "reset password please",                                                  False, 7),
    ("T009", "Cannot log in. Tried resetting password twice but emails never arrived. "
             "Checked spam folder. Account may have been locked.",                      True,  8),
    ("T010", "password issue",                                                         False, 9),
]

for uid, text, has_reason, days in cr31_transcripts:
    ts = (base_ts + timedelta(days=days)).isoformat()
    store.add_correction(
        "contact_reason", uid=uid, text=text,
        original={"contact_reason_primary_code": "CR1.2"},
        corrected={"contact_reason_primary_code": "CR3.1"},
        reason="Password/account recovery issue, not login blocked" if has_reason else "",
    )
    # Backpatch timestamp
    corrections = store.load_corrections("contact_reason")
    corrections[-1]["timestamp"] = ts
    store._cache.pop("contact_reason", None)
    path = store._path("contact_reason")
    with open(path, "w", encoding="utf-8") as f:
        for c in corrections:
            f.write(json.dumps(c) + "\n")

# Also add two corrections for different codes
store.add_correction(
    "contact_reason", uid="B001",
    text="Customer received wrong invoice amount. Charged twice for same subscription. "
         "Original charge was $49/month but billed $98. Requested refund for duplicate.",
    original={"contact_reason_primary_code": "CR2.1"},
    corrected={"contact_reason_primary_code": "CR2.3"},
    reason="Billing dispute not general billing enquiry",
)
store.add_correction(
    "contact_reason", uid="A001",
    text="Feature not available in the free tier. User trying to export data "
         "but export button is greyed out. Needs upgrade to access export functionality.",
    original={"contact_reason_primary_code": "CR4.1"},
    corrected={"contact_reason_primary_code": "CR4.2"},
    reason="Feature access question not a bug",
)

stats = store.get_stats()
print(f"Corrections loaded: {stats}")
print(f"Total: {sum(stats.values())} corrections across {len(stats)} module(s)")

# Show what 'recent' strategy does
print("\n─── recent (max_examples=5): selects these UIDs ───")
recent = store._select_examples(store.load_corrections("contact_reason"), 5, "recent")
for c in recent:
    print(f"  uid={c['uid']:5s}  code={c['corrected'].get('contact_reason_primary_code')}  "
          f"reason={'yes' if c['reason'] else 'no ':3s}  text='{c['text'][:50]}'")

# Show what 'diverse' strategy does
print("\n─── diverse (max_examples=5): selects these UIDs ───")
diverse = store._select_examples(store.load_corrections("contact_reason"), 5, "diverse")
for c in diverse:
    print(f"  uid={c['uid']:5s}  code={c['corrected'].get('contact_reason_primary_code')}  "
          f"reason={'yes' if c['reason'] else 'no ':3s}  text='{c['text'][:50]}'")

## 2 · Stage 1 — Content-aware pool building

### The `content_score` formula

Raw text length is a poor signal — long transcripts can be mostly boilerplate (templated agent responses, email chains, repeated troubleshooting scripts). Instead, the score uses the **product** of two complementary measures:

```
tokens_all    = text.lower().split()              # all tokens
unique        = len(set(tokens_all))              # distinct tokens
total         = max(len(tokens_all), 1)

diversity_ratio  = unique / total                  # penalises repetition
capped_unique    = min(unique, 80) / 80            # penalises trivially short texts
content_score    = diversity_ratio * capped_unique # high only when BOTH are high
```

The product means neither dimension can "fake" a high score alone:
- Long boilerplate: many tokens → low diversity → low product
- 3-word transcript: high diversity → near-zero capped_unique → low product
- Rich, varied transcript: both measures are high → high product

### Stage 1 overall score

```
score = 0.4 * has_reason + 0.3 * content_score + 0.3 * recency
```

One representative per distinct corrected code is kept (highest scorer in each group).

In [ ]:
def content_score(text: str) -> dict:
    """Show the content_score breakdown for a given text."""
    tokens_all = re.sub(r"[^a-z0-9\s]", "", text.lower()).split()
    unique = len(set(tokens_all))
    total  = max(len(tokens_all), 1)
    div    = unique / total
    cap    = min(unique, 80) / 80
    score  = div * cap
    return {"total_tokens": total, "unique_tokens": unique,
            "diversity_ratio": round(div, 3), "capped_unique": round(cap, 3),
            "content_score": round(score, 3)}

examples = {
    "Boilerplate (long, repetitive)": (
        "Customer called. Thank you for contacting us. I understand you are experiencing "
        "difficulties. Please follow these steps. Thank you for contacting us again. "
        "I understand. Please follow these steps. Thank you for your patience. "
        "I understand your concern. Please follow these steps."
    ),
    "Too short": "password issue",
    "Short but specific": "Account locked after 3 failed attempts, recovery email not working",
    "Rich medium (T006)": (
        "User cannot access account. Password reset link expired after 24h. "
        "Secondary email also unavailable. Escalated to Tier 2 for manual reset."
    ),
    "Templated troubleshooting": (
        "Thank you for contacting support. I understand you are experiencing "
        "difficulties. Please follow these steps: 1. Go to login page "
        "2. Click forgot password 3. Check email 4. Click reset link"
    ),
    "Invoice dispute (B001)": (
        "Customer received wrong invoice amount. Charged twice for same subscription. "
        "Original charge was $49/month but billed $98. Requested refund for duplicate."
    ),
}

rows = []
for name, text in examples.items():
    row = {"transcript_type": name}
    row.update(content_score(text))
    rows.append(row)

df = pd.DataFrame(rows).set_index("transcript_type")
print("Content score breakdown:\n")
print(df.to_string())

In [ ]:
# Build the Stage 1 pool
pool = store.build_smart_pool("contact_reason")

print(f"Pool: {len(pool)} entries (one per distinct corrected code)\n")
print(f"{'uid':6s}  {'code':8s}  {'stage1_score':12s}  {'has_reason':10s}  {'text (first 70 chars)'}")
print("─" * 100)
for entry in sorted(pool, key=lambda x: x.get("_stage1_score", 0), reverse=True):
    code     = entry["corrected"].get("contact_reason_primary_code", "?")
    score    = entry.get("_stage1_score", 0)
    has_r    = "yes" if entry.get("reason") else "no"
    text_pre = entry["text"][:70].replace("\n", " ")
    print(f"  {entry['uid']:5s}  {code:8s}  {score:12.3f}  {has_r:10s}  {text_pre}")

print()
print("Note: all 10 CR3.1 corrections collapsed to 1 representative (T006 wins)")
cr31 = next(e for e in pool if e["corrected"].get("contact_reason_primary_code") == "CR3.1")
print(f"  Selected: uid={cr31['uid']}  reason='{cr31.get('reason', '')[:60]}'")
print(f"  Text: '{cr31['text'][:100]}'")

## 3 · Stage 2 — Per-batch Jaccard reranking

From the candidate pool, each batch gets the examples most similar to its own transcripts.

**Similarity metric:** Jaccard token-overlap between each correction's text and each record in the batch:

```
overlap(correction, record) = |tokens(correction) ∩ tokens(record)|
                              / max(|tokens(correction)|, |tokens(record)|)

stage2_score(correction, batch) = max overlap across all records in the batch
```

Using `max` (not `mean`) means a **single strong match** within the batch is enough to surface the correction — you don't need every record to match.

In [ ]:
# Three synthetic batches with different topic profiles
batch_password = [
    {"uid": "P1", "text": "Cannot log in after password reset. Email link expired."},
    {"uid": "P2", "text": "Account locked. Recovery email not received. Tried twice."},
    {"uid": "P3", "text": "Forgot password and username. Secondary email also inaccessible."},
]
batch_billing = [
    {"uid": "B1", "text": "Charged twice on my invoice. Need a refund for the duplicate payment."},
    {"uid": "B2", "text": "My subscription fee was $49 but invoice shows $98. Wrong amount."},
    {"uid": "B3", "text": "Invoice discrepancy. Contacted billing team but no resolution yet."},
]
batch_feature = [
    {"uid": "F1", "text": "Export button is greyed out in my account. Can't download my data."},
    {"uid": "F2", "text": "Feature not available on free tier. Unsure if I need to upgrade."},
    {"uid": "F3", "text": "Cannot access the export functionality. Button is disabled."},
]

def show_reranked(batch_name: str, batch: list, pool: list, max_examples: int = 5):
    print(f"\n{'='*70}")
    print(f"Batch: {batch_name}")
    print(f"  Records: {[r['uid'] for r in batch]}")
    reranked = store.rerank_pool_for_batch(pool, batch, max_examples)
    print(f"  Top {len(reranked)} selected (of {len(pool)} pool entries):")
    print(f"    {'uid':5s}  {'code':8s}  {'stage2_score':12s}  {'stage1_score':12s}  {'text (60 chars)'}")
    print(f"    {'─'*80}")
    for e in reranked:
        code = e["corrected"].get("contact_reason_primary_code", "?")
        s2   = e.get("_stage2_score", 0)
        s1   = e.get("_stage1_score", 0)
        text = e["text"][:60].replace("\n", " ")
        print(f"    {e['uid']:5s}  {code:8s}  {s2:12.3f}  {s1:12.3f}  {text}")

show_reranked("Password / Account Recovery", batch_password, pool)
show_reranked("Billing Dispute",             batch_billing,  pool)
show_reranked("Feature Access",              batch_feature,  pool)

print()
print("Observation: each batch gets the examples most relevant to its own content.")
print("The billing batch does NOT get the password reset example at the top.")

## 4 · What the LLM actually sees — the few-shot prompt block

After reranking, `build_few_shot_prompt_for_batch` formats the selected corrections into the system prompt block.

In [ ]:
reranked_billing = store.rerank_pool_for_batch(pool, batch_billing, max_examples=5)
prompt_block = store.build_few_shot_prompt_for_batch(
    reranked_billing, max_text_chars=300, max_total_chars=4000
)

print("Few-shot block for billing batch:")
print("─" * 70)
print(prompt_block)
print("─" * 70)
print(f"\nTotal chars: {len(prompt_block)}")

## 5 · Strategy comparison

Same corrections, same batch — which examples does each strategy select?

In [ ]:
corrections = store.load_corrections("contact_reason")
target_batch = batch_billing  # use the billing batch as the reference

print(f"Target batch: {[r['uid'] for r in target_batch]}")
print(f"Total corrections in store: {len(corrections)}")
print()

strategies = ["recent", "diverse", "frequent"]
for strat in strategies:
    selected = store._select_examples(corrections, 5, strat)
    codes  = [c["corrected"].get("contact_reason_primary_code") for c in selected]
    uids   = [c["uid"] for c in selected]
    print(f"Strategy: {strat:10s}  → {len(selected)} examples  uids={uids}  codes={codes}")

print()
pool_smart = store.build_smart_pool("contact_reason")
reranked   = store.rerank_pool_for_batch(pool_smart, target_batch, max_examples=5)
codes  = [c["corrected"].get("contact_reason_primary_code") for c in reranked]
uids   = [c["uid"] for c in reranked]
s2     = [round(c.get("_stage2_score", 0), 2) for c in reranked]
print(f"Strategy: smart       → {len(reranked)} examples  uids={uids}  codes={codes}")
print(f"          stage2_scores (Jaccard vs billing batch): {s2}")
print()
print("Smart correctly surfaces B001 (billing correction) first.")
print("Static strategies have no way to know which batch this is for.")

## 6 · Pipeline integration

### Via `RunConfig`

Set `feedback_strategy="smart"` in your `RunConfig`:

In [ ]:
from rca_pipeline.config import RunConfig

cfg = RunConfig(
    input_file="data/transcripts.csv",
    output_file="runs/output/enriched.xlsx",
    taxonomy_dir="taxonomies",
    text_col="conversation",
    id_col="uid",
    modules=["contact_reason"],
    # ── Smart feedback ────────────────────────────────────────
    feedback_dir="feedback",
    feedback_max_examples=5,      # max examples per batch (Stage 2 cap)
    feedback_strategy="smart",    # ← enable two-stage selection
)

print(f"feedback_strategy : {cfg.feedback_strategy}")
print(f"feedback_max_examples: {cfg.feedback_max_examples}")
print(f"feedback_dir      : {cfg.feedback_dir}")

In [ ]:
# ── Via a run profile (JSON) ──────────────────────────────────────────────
profile = {
    "input":              "data/transcripts.csv",
    "output":             "runs/output/enriched.xlsx",
    "text_col":           "conversation",
    "taxonomy_dir":       "taxonomies",
    "feedback_dir":       "feedback",
    "feedback_max_examples": 5,
    "feedback_strategy":  "smart",  # ← enable two-stage selection
    "batch_size":         10,
    "concurrency":        4,
    "model":              "azure/dsaic-llm-general",
}

profile_path = Path("profiles/smart_feedback_example.json")
profile_path.parent.mkdir(exist_ok=True)
profile_path.write_text(json.dumps(profile, indent=2))
print(f"Profile written to: {profile_path}")
print()
print("To run:")
print(f"  rca run --profile {profile_path}")

### What to expect in the logs

When `strategy="smart"`, the pipeline logs:

```
INFO | pipeline | Feedback smart pool: 3 candidates for contact_reason
```

At batch-dispatch time (DEBUG level), each batch's selected examples are different.
To see per-batch selection, enable DEBUG logging:

```python
import logging
logging.getLogger("feedback").setLevel(logging.DEBUG)
```

## 7 · Bulk import from a reviewed spreadsheet

After a classification run, export the review queue, correct classifications in Excel, then import all corrections at once.

**Expected format:** Your review spreadsheet should have the original classification columns plus `<field>_corrected` columns for rows that were corrected. Only rows where `<field>_corrected` differs from `<field>` are imported.

In [ ]:
# ── Example: import corrections from a reviewed spreadsheet ──────────────

# Uncomment and adjust paths when you have a reviewed file:
# store = FeedbackStore("feedback/")
# count = store.import_from_review_file(
#     filepath="runs/output/review_queue_corrected.xlsx",
#     module="contact_reason",
#     text_col="conversation",          # column containing the transcript
#     uid_col="uid",                    # column containing the record ID
#     corrected_suffix="_corrected",    # suffix on corrected columns
# )
# print(f"Imported {count} corrections")

# After importing, check the pool quality:
# pool = store.build_smart_pool("contact_reason")
# print(f"Pool size: {len(pool)} (one per distinct corrected code)")
# for e in pool:
#     print(f"  {e['corrected']}  score={e['_stage1_score']:.3f}  uid={e['uid']}")

print("Uncomment the cells above after you have a reviewed corrections spreadsheet.")
print("The correction file format:")
print("  - contact_reason_primary_code          (original model output)")
print("  - contact_reason_primary_code_corrected (human correction — blank if correct)")
print("  - correction_reason                    (optional freetext explanation)")

## 8 · Diagnostic: inspect pool and per-batch selections

This section shows how to audit what the smart strategy is doing without running the full pipeline.

In [ ]:
# Full pool audit
pool = store.build_smart_pool("contact_reason")

print("=" * 80)
print(f"STAGE 1 POOL  ({len(pool)} entries)")
print("=" * 80)
for e in sorted(pool, key=lambda x: x.get("_stage1_score", 0), reverse=True):
    code  = list(e["corrected"].values())[0]
    score = e.get("_stage1_score", 0)
    tokens_all = re.sub(r"[^a-z0-9\s]", "", e["text"].lower()).split()
    unique = len(set(tokens_all))
    total  = max(len(tokens_all), 1)
    cs = (unique / total) * (min(unique, 80) / 80)
    print(f"  uid={e['uid']:5s}  {code:6s}  stage1={score:.3f}  "
          f"content={cs:.3f}  reason={'✓' if e.get('reason') else '✗'}  "
          f"text='{e['text'][:55]}'")

print()
# Simulate what a real pipeline would do for 4 sample batches
sample_batches = [
    ("batch-0 (password)",  batch_password),
    ("batch-1 (billing)",   batch_billing),
    ("batch-2 (feature)",   batch_feature),
    ("batch-3 (mixed)",     batch_password[:1] + batch_billing[:1] + batch_feature[:1]),
]

print("=" * 80)
print("STAGE 2 SELECTIONS PER BATCH")
print("=" * 80)
for batch_name, batch in sample_batches:
    reranked = store.rerank_pool_for_batch(pool, batch, max_examples=3)
    selected_codes = [list(e["corrected"].values())[0] for e in reranked]
    s2_scores = [round(e.get("_stage2_score", 0), 3) for e in reranked]
    uids = [e["uid"] for e in reranked]
    print(f"  {batch_name:30s} → codes={selected_codes}  uids={uids}  jaccard={s2_scores}")

## Summary

| | `recent` | `diverse` | `frequent` | `smart` |
|---|---|---|---|---|
| Per-batch selection | No | No | No | **Yes** |
| Handles 10 same-code corrections | Wastes 5 slots | 1 example (arbitrary) | 1 example | **Best representative** |
| Penalises boilerplate | No | No | No | **Yes (content_score)** |
| Extra API calls | 0 | 0 | 0 | **0** |
| Startup cost | O(N) | O(N) | O(N) | **O(N)** |
| Per-batch cost | O(1) | O(1) | O(1) | **O(pool × batch)** — fast string ops |

**Recommended workflow:**
1. Start with `strategy="diverse"` or `strategy="recent"` — no corrections needed, quick to set up
2. After accumulating 10+ corrections across multiple codes, switch to `strategy="smart"`
3. Use `build_smart_pool()` + the diagnostic cells above to audit what's being injected
4. Add `correction_reason` when importing corrections — it's the highest-weight signal in Stage 1

**Improvement tip:** The quality of Stage 1 selection depends heavily on the `reason` field. Reviewers who explain *why* a correction was made produce far more useful few-shot examples than those who only change the code.